# Gene to Protein Analysis

This notebook focuses on analyzing individual gene mutations and their effects on protein sequences using the Ensembl VEP API.

In [11]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
os.chdir(os.path.dirname(os.path.abspath('.')))

import json
from datetime import datetime

import src.utils as utils
import src.config as config
import src.gprofiler as gp
import src.vep_pipeline as vp
import src.vep_analysis as va
import src.vep_metrics as vm
import src.proteingym as pg 
import src.ensembl_rest as er
import src.biopython as bp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Get Gene and Variant Information

First, let's create functions to get information about a gene and its variants.

In [14]:
# Create client
client = er.get_ensembl_client()

# Initialize results dictionary
results = {
    "query_info": {
        "gene": "HBB",
        "variant_id": "rs334",
        "timestamp": datetime.now().isoformat()
    },
    "gene_info": {},
    "sequences": {},
    "variant_effects": {},
    "variation_details": {}
}

# Look up gene
gene_info = client.symbol_lookup(species='homo_sapiens', symbol='HBB')
results["gene_info"] = {
    "id": gene_info.get('id'),
    "location": f"{gene_info.get('seq_region_name')}:{gene_info.get('start')}-{gene_info.get('end')}"
}

# Get variant effects for sickle cell mutation
variant_id = 'rs334'  # This is the sickle cell variant
vep_results = er.get_vep(
    ids=variant_id,
    species='homo_sapiens',
    params={
        'canonical': 1,
        'protein': 1,
        'uniprot': 1,
        'domains': 1,
        'numbers': 1,
        'hgvs': 1
    }
)

if isinstance(vep_results, dict):
    for variant_id, variant_results in vep_results.items():
        results["variant_effects"][variant_id] = []
        
        for result in variant_results:
            for tc in result.get('transcript_consequences', []):
                if tc.get('canonical') == 1:  # Get canonical transcript info
                    transcript_id = tc.get('transcript_id')
                    protein_id = tc.get('protein_id')
                    
                    transcript_data = {
                        "transcript_id": transcript_id,
                        "protein_id": protein_id,
                        "uniprot_id": tc.get('swissprot', [''])[0],
                        "gene_sequence": None,
                        "protein_sequence": None,
                        "variant_effect": {
                            "impact": tc.get('impact'),
                            "hgvs_p": tc.get('hgvsp'),
                            "position": f"{tc.get('protein_start')}-{tc.get('protein_end')}",
                            "amino_acid_change": tc.get('amino_acids')
                        }
                    }
                    
                    # Get gene sequence if transcript_id exists
                    if transcript_id:
                        try:
                            gene_seq_response = client.sequence_id(species='homo_sapiens', id=transcript_id)
                            if isinstance(gene_seq_response, dict):
                                transcript_data["gene_sequence"] = gene_seq_response.get('seq', '')
                            else:
                                transcript_data["gene_sequence"] = str(gene_seq_response)
                        except Exception as e:
                            transcript_data["gene_sequence_error"] = str(e)
                    
                    # Get protein sequence if protein_id exists
                    if protein_id:
                        try:
                            protein_seq_response = client.sequence_id(species='homo_sapiens', id=protein_id)
                            if isinstance(protein_seq_response, dict):
                                transcript_data["protein_sequence"] = protein_seq_response.get('seq', '')
                            else:
                                transcript_data["protein_sequence"] = str(protein_seq_response)
                        except Exception as e:
                            transcript_data["protein_sequence_error"] = str(e)
                    
                    results["variant_effects"][variant_id].append(transcript_data)

# Get variation details
variation_info = er.get_variation(
    variant_id=variant_id,
    species='homo_sapiens'
)
if variation_info:
    results["variation_details"] = {
        "maf": variation_info.get('MAF'),
        "clinical_significance": variation_info.get('clinical_significance')
    }

# Save to JSON file in the results directory
output_file = os.path.join('/home/caom/VEP_protein/results', f"hbb_variant_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults have been saved to: {output_file}")

Getting variant info:   0%|          | 0/1 [00:00<?, ?it/s]


Results have been saved to: /home/caom/VEP_protein/results/hbb_variant_analysis_20250319_121008.json
